## IronKaggle - mini project
adapting original notebook1 to deploy API

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import necessary libraries for data preprocessing and modeling
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
import pickle


## Read data

In [2]:
sales_df = pd.read_csv('sales.csv')

### EDA
removed in notebook2, still available on notebook1

## Notes

- Drop First column (Unamed:0) when reading the file. Could be indices.
- `store_ID` : Unique ids of the shop. Hass 1115 unique values. Numerical values could be indicate a relation or order between the stores. Could be dropped but we might loss some information about the shop.
- There are two columns with datatypes str, which need to be converted to numerical type.
  - `state_holiday` is a ctegorical column with 4 categories. One-Hot encoding needed.
- `day_of_week` is a categorical column with ordinal values. We can keep the ordinal values as the days of a week are related and sequence of the days in a week do not change. No change required for this column.
-  Convaert date format into thre columns of year, month and date.
-  No missing values

In [3]:
# Drop the Unamed column
sales_df.drop(columns='Unnamed: 0', inplace=True)

#convert object date column to Date time
sales_df['date'] = pd.to_datetime(sales_df['date'])
# Split the date column using dt accessor
sales_df['year'] = sales_df['date'].dt.year
sales_df['month'] = sales_df['date'].dt.month
sales_df['day'] = sales_df['date'].dt.day

# drop the date column
sales_df.drop(columns='date', inplace=True)

sales_df = sales_df[sales_df["open"] == 1].drop(columns="open")

sales_df

,store_ID,day_of_week,nb_customers_on_day,promotion,state_holiday,school_holiday,sales,year,month,day
0,366,4,517,0,0,0,4422,2013,4,18
1,394,6,694,0,0,0,8297,2015,4,11
2,807,4,970,1,0,0,9729,2013,8,29
3,802,2,473,1,0,0,6513,2013,5,28
4,726,4,1068,1,0,0,10882,2013,10,10
...,...,...,...,...,...,...,...,...,...,...
640833,77,6,701,0,0,0,8219,2015,1,17
640835,409,6,483,0,0,0,4553,2013,10,26
640836,97,1,987,1,0,0,12307,2014,4,14
640837,987,1,925,0,0,0,6800,2014,7,7


- There is not much correlation between the features
- We can notice correlation between the sales (target) and the features. This is a good indication. This means that these features are important for the sales.

- The distribution of values for each features suggests that sales and number of customers where 0 when the store was closed.
- Looking at the descriptive statistics for both caes (open and closed store), we see a difference in the values.
- It is safe to infer that when the store is closed there are no sale, so when can drop the rows when the store was open. Also, we can see that the store was closed for ~17% of the time.

In [4]:
sales_df.state_holiday.value_counts(normalize=True)

state_holiday
0    0.998912
a    0.000806
b    0.000192
c    0.000090
Name: proportion, dtype: float64

## Data splitting

In [5]:
from sklearn.model_selection import train_test_split


In [6]:
# Extract the data in features and target
X = sales_df.drop(columns='sales')
y = sales_df['sales']

In [7]:
# Split data into Train, Validation, Test sets (70, 15, 15)
# first split: train (70%) and temporary set (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# second split: validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Check the size of each split
print('X_train shape:', X_train.shape)
print('X_val shape:  ', X_val.shape)
print('X_test shape: ', X_test.shape)

X_train shape: (372411, 9)
X_val shape:   (79802, 9)
X_test shape:  (79803, 9)


## Data cleaning

## Feature engineering

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
# Features to transform
categorical_features = ["state_holiday"]

numeric_features = [
    "store_ID",
    "nb_customers_on_day",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
    ],
    remainder="passthrough"
)


In [10]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=200,
                max_depth=10,
                random_state=42
            ),
        ),
    ]
)

## Baseline model training

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# Train the model on the training data (ignore returned fitted model)
pipeline.fit(X_train, y_train)

# Compute the predictions on test sets
y_pred = pipeline.predict(X_test)

   
    

- We will choose the XGBoost model to tune.

# Save model

In [12]:
import pickle

In [13]:
# save the model
with open("Final_Model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

## Test Load model

In [14]:
with open("Final_Model.pkl", "rb") as f:
    model = pickle.load(f)

sample = X_test.iloc[[0]]

prediction = model.predict(sample)

print(prediction)

[6824.315]
